In [10]:
import json
import pickle
from pathlib import Path

In [11]:
INPUT_JSON   = Path("../retrieval/locisimiles/results_k100.json")  
OUTPUT_JSON  = Path("../retrieval/locisimiles/retrieval_results_locisimiles_k100.json")
OUTPUT_PKL   = Path("../retrieval/locisimiles/retrieval_results_locisimiles_k100.pkl")

MODEL_NAME   = "locisimiles"                     
INDEX_TYPE   = "LocíSimiles standard pipeline"   
MAX_K        = 100
K_VALUES     = [5, 10, 20]

In [ ]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Loaded {len(raw)} queries")
# Preview query
first_key = next(iter(raw))
print(f"\nExample query id : {first_key}")
print(f"Candidates       : {len(raw[first_key])}")
print("First candidate  :", raw[first_key][0])

Loaded 19466 queries

Example query id : 10015_sent_0
Candidates       : 100
First candidate  : {'source_id': 'tlg2018.tlg002.perseus-grc2_window_554', 'source_text': 'Συμμάχου τοῦ ἑρμηνέως. ΙΗ Περὶ Ἀμβροσίου. ΙΘ Ὅσα περὶ Ὠριγένους μνημονεύεται. Κ Ὅσοι τῶν τηνικάδε φέρονται λόγοι. ΚΑ Ὅσοι κατὰ τούσδε ἐπίσκοποι ἐγνωρίζοντο. ΚΒ Ὅσα τῶν Ἱππολύτου εἰς ἡμᾶς ἦλθεν. ΚΓ Περὶ τῆς Ὠριγένους σπουδῆς καὶ ὡς τοῦ ἐκκλησιαστικοῦ πρεσβείου ἠξιώθη. ΚΔ Τίνα ἐπὶ τῆς Ἀλεξανδρείας ἐξηγήσατο. ΚΕ Ὅπως τῶν ἐνδιαθήκων γραφῶν ἐμνημόνευσεν. ΚΣ Ὅπως αὐτὸν ἑώρων οἱ ἐπίσκοποι. ΚΖ Ὡς Ἡρακλᾶς τὴν Ἀλεξανδρέων ἐπισκοπὴν διεδέξατο. ΚΗ Περὶ τοῦ κατὰ Μαξιμῖνον διωγμοῦ. ΚΘ Περὶ Φαβιανοῦ ὡς Ῥωμαίων ἐπίσκοπος ἐκ θεοῦ παραδόξως ἀνεδείχθη. Λ Ὅσοι γεγόνασιν Ὠριγένους φοιτηταί. ΛΑ Περὶ Ἀφρικανοῦ. ΛΒ Τίνα Ὠριγένης ἐν Καισαρείᾳ τῆς Παλαιστίνης ἐξηγήσατο. ΛΓ Περὶ τῆς Βηρύλλου παρατροπῆς. ΛΔ Τὰ κατὰ Φίλιππον. ΛΕ Ὡς Διονύσιος Ἡρακλᾶ τὴν ἐπισκοπὴν διεδέξατο. ΛΣ Ὅσα ἄλλα ἐσπούδαστο τῷ Ὠριγένει. ΛΖ Περὶ τῆς τῶν Ἀράβων διαστάσεως. ΛΗ Περ

In [13]:
retrieval_results = {} 
total_candidates = 0

for query_id, candidates in raw.items():
    transformed = []
    for cand in candidates:
        transformed.append((cand["source_id"], cand["candidate_score"]))
        total_candidates += 1
    retrieval_results[query_id] = transformed

In [14]:
# JSON 
output_data = {
    "metadata": {
        "model_name": MODEL_NAME,
        "total_queries": len(retrieval_results),
        "total_candidates": total_candidates,
        "max_k": MAX_K,
        "k_values": K_VALUES,
        "index_type": INDEX_TYPE,
    },
    "retrieval_results": {
        query_id: [
            {
                "candidate_id": cand_id,
                "similarity": score,
                "rank": rank,
            }
            for rank, (cand_id, score) in enumerate(candidates, 1)
        ]
        for query_id, candidates in retrieval_results.items()
    }
}

In [15]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

# PKL — tuples only, same as FAISS
with open(OUTPUT_PKL, "wb") as f:
    pickle.dump(retrieval_results, f)

In [ ]:
# save judgements for threshold analysis
OUTPUT_PKL_judge   = Path("../retrieval/locisimiles/retrieval_results_locisimiles_judgement_k100.pkl")

In [ ]:
retrieval_results = {} 
total_candidates = 0

for query_id, candidates in raw.items():
    transformed = []
    for cand in candidates:
        transformed.append((cand["source_id"], cand["judgment_score"]))
        total_candidates += 1
    retrieval_results[query_id] = transformed

# PKL — tuples only
with open(OUTPUT_PKL_judge, "wb") as f:
    pickle.dump(retrieval_results, f)